## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# Euler's Method and RK2

**Learning objectives**

By the end of this notebook you will be able to:

- Explain what an ordinary differential equation (ODE) is and identify its components
- Implement Forward Euler's method and analyse how step size affects accuracy
- Describe how Backward Euler and the Trapezoidal method achieve unconditional stability
- Implement two explicit second-order methods: RK2 Heun and RK2 Midpoint

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ode_hints import show_hint
from ode_verify import check_forward_euler, check_rk2_heun, check_rk2_midpoint

## What Is an ODE?

An **ordinary differential equation** (ODE) relates a function to its derivatives. The general first-order form is:

$$\frac{dy}{dt} = f(t,\, y), \qquad y(t_0) = y_0$$

The word *ordinary* means there is only one independent variable ($t$). The pair $(t_0, y_0)$ is the **initial condition** — the known starting point.

### The Test Equation

Throughout this notebook we study the **test equation**:

$$\frac{dy}{dt} = \lambda y, \qquad y(0) = y_0$$

where $\lambda$ is a constant. This equation is the standard benchmark for analysing numerical ODE solvers — it is simple enough to solve exactly, yet rich enough to reveal the accuracy and stability properties of every method we introduce.

### Running Example

We set $\lambda = -2$ and $y_0 = 1$:

$$\frac{dy}{dt} = -2y, \qquad y(0) = 1$$

**Analytical solution** (by separation of variables):

$$\frac{dy}{y} = -2\,dt \;\Longrightarrow\; \ln|y| = -2t + C \;\Longrightarrow\; y(t) = e^{-2t}$$

since $y(0) = 1$ gives $C = 0$.

Most ODEs have no closed-form solution, so we turn to **numerical methods**: chop the interval into small steps of width $h$, start from the known initial value, and march forward one step at a time.

In [ ]:
# Running example: dy/dt = -2y, y(0) = 1, solution y = exp(-2t)
def f(t, y):
    """Right-hand side of the running example ODE."""
    return -2 * y

def y_exact(t):
    """Analytical solution of the running example."""
    return np.exp(-2 * t)

# Global parameters
t_span = (0, 3)
y0 = 1
t_dense = np.linspace(0, 3, 300)

# Plot the analytical solution
plt.figure(figsize=(7, 4))
plt.plot(t_dense, y_exact(t_dense), color="steelblue", linewidth=2)
plt.xlabel("t")
plt.ylabel("y")
plt.title("Analytical solution: $y(t) = e^{-2t}$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1. Forward Euler's Method

At the current point $(t_n, y_n)$, the ODE tells us the **slope**: $f(t_n, y_n)$. If we follow this slope (i.e. the tangent line) for a small distance $h$, we arrive at an approximation for the next point.

From the Taylor expansion:

$$y(t + h) = y(t) + h \cdot y'(t) + \mathcal{O}(h^2)$$

Dropping the $\mathcal{O}(h^2)$ term and replacing $y'$ with $f$ gives the **Forward Euler update rule**:

$$y_{n+1} = y_n + h \cdot f(t_n,\, y_n)$$

This is a **first-order** method: the local truncation error at each step is $\mathcal{O}(h^2)$, and the accumulated global error over the whole interval is $\mathcal{O}(h)$.

### 1.1 Implementation

The algorithm:

1. Create an array of time points from $t_0$ to $t_{\text{end}}$ with spacing $h$.
2. Initialise $y_0$.
3. Loop: at each step, apply $y_{n+1} = y_n + h \cdot f(t_n, y_n)$.
4. Return both arrays.

Fill in the loop body below.

In [ ]:
def forward_euler(f, t_span, y0, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0

    # ===== YOUR CODE BELOW =====



    # ===== YOUR CODE ABOVE =====
    return t, y

In [ ]:
show_hint("forward_euler")

In [ ]:
check_forward_euler(forward_euler)

### 1.2 Numerical vs Analytical

Let us use the function you just wrote to compare the numerical solution against the exact answer.

In [ ]:
# Complete code — just run this cell
t_euler, y_euler = forward_euler(f, t_span, y0, h=0.2)

# Step-by-step table (first 10 steps)
print(f"{'t':>6s}  {'y_euler':>10s}  {'y_exact':>10s}  {'error':>10s}")
print("-" * 42)
for i in range(min(11, len(t_euler))):
    exact_val = y_exact(t_euler[i])
    err = abs(y_euler[i] - exact_val)
    print(f"{t_euler[i]:6.2f}  {y_euler[i]:10.6f}  {exact_val:10.6f}  {err:10.6f}")

# Comparison plot
plt.figure(figsize=(7, 4))
plt.plot(t_dense, y_exact(t_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-2t}$")
plt.plot(t_euler, y_euler, "o-", color="crimson", markersize=4,
         label="Forward Euler (h=0.2)")
plt.xlabel("t")
plt.ylabel("y")
plt.title("Forward Euler vs analytical solution")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.3 Step Size and Accuracy

Smaller $h$ means the tangent approximation is used over a shorter distance, so each step is more accurate — but more steps are needed. Forward Euler's global error is $\mathcal{O}(h)$: halving $h$ roughly halves the error.

A **log–log plot** of error versus $h$ should show a straight line with slope $\approx 1$, confirming first-order convergence.

In [ ]:
# Complete code — just run this cell
step_sizes = [0.5, 0.1, 0.01]
errors = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: solutions for each step size
axes[0].plot(t_dense, y_exact(t_dense), color="steelblue", linewidth=2,
             label="Analytical: $y = e^{-2t}$")
for h_val, col in zip(step_sizes, ["crimson", "forestgreen", "darkorange"]):
    t_h, y_h = forward_euler(f, t_span, y0, h_val)
    axes[0].plot(t_h, y_h, "o-", color=col, markersize=3, label=f"h={h_val}")
    idx = np.argmin(np.abs(t_h - 1.0))
    errors.append(abs(y_h[idx] - y_exact(t_h[idx])))

axes[0].set_xlabel("t")
axes[0].set_ylabel("y")
axes[0].set_title("Forward Euler with different step sizes")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: log-log error plot
axes[1].loglog(step_sizes, errors, "ko-", label="Measured error at t=1")
h_ref = np.array([0.5, 0.1, 0.01])
axes[1].loglog(h_ref, 0.5 * h_ref, "--", color="grey", label="Slope 1 (reference)")
axes[1].set_xlabel("Step size h")
axes[1].set_ylabel("Absolute error")
axes[1].set_title("Convergence: error vs step size")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Beyond Forward Euler

Forward Euler has two limitations:

1. **Low accuracy.** It is only first-order — halving $h$ merely halves the error.

2. **Stability.** For some step sizes the numerical solution oscillates and diverges, even though the true solution decays smoothly.

**Deriving the stability condition.** For the test equation $\frac{dy}{dt} = \lambda y$ with $\lambda < 0$, Forward Euler gives:

$$y_{n+1} = y_n + h \lambda\, y_n = (1 + h\lambda)\, y_n$$

After $n$ steps: $y_n = (1 + h\lambda)^n \, y_0$. For the solution to decay rather than blow up, we need the **amplification factor** to satisfy:

$$|1 + h\lambda| < 1$$

Since $\lambda < 0$ and $h > 0$, the product $h\lambda$ is negative, so $1 + h\lambda < 1$ automatically. The binding constraint is:

$$1 + h\lambda > -1 \;\Longrightarrow\; h\lambda > -2 \;\Longrightarrow\; h < \frac{2}{|\lambda|}$$

For our running example ($\lambda = -2$), the limit is $h < 1$. The plot below uses the faster-decaying equation $\frac{dy}{dt} = -10y$ with $h = 0.25$ (limit is $h < 0.2$).

In [ ]:
# Complete code — just run this cell
# Forward Euler on a fast-decay equation with h above the stability limit
f_fast = lambda t, y: -10 * y
y_fast_exact = lambda t: np.exp(-10 * t)
t_span_fast = (0, 2)
h_fast = 0.25  # Stability limit is h < 2/10 = 0.2

t_fe_fast, y_fe_fast = forward_euler(f_fast, t_span_fast, 1.0, h_fast)
t_fast_dense = np.linspace(0, 2, 300)

plt.figure(figsize=(8, 5))
plt.plot(t_fast_dense, y_fast_exact(t_fast_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-10t}$")
plt.plot(t_fe_fast, y_fe_fast, "o-", color="crimson", markersize=5,
         label=f"Forward Euler (h={h_fast})")
plt.axhline(0, color="grey", linewidth=0.5)
plt.xlabel("t")
plt.ylabel("y")
plt.title("Forward Euler blows up when $h > 2/|\\lambda|$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Stability limit: h < 2/10 = 0.2")
print(f"Step size used:  h = {h_fast} — exceeds the limit.")
print(f"Amplification factor: |1 + h*lambda| = |1 + 0.25*(-10)| = 1.5 > 1")

### 2.1 Backward Euler (Implicit)

Instead of using the slope at the **current** point, Backward Euler uses the slope at the **next** point:

$$y_{n+1} = y_n + h \cdot f(t_{n+1},\, y_{n+1})$$

Notice that $y_{n+1}$ appears on **both sides** — this is what makes the method **implicit**.

**Algebraic solution for the test equation.** When $f(t,y) = \lambda y$:

$$y_{n+1} = y_n + h\lambda\, y_{n+1} \;\Longrightarrow\; y_{n+1}(1 - h\lambda) = y_n \;\Longrightarrow\; y_{n+1} = \frac{y_n}{1 - h\lambda}$$

The amplification factor is $\frac{1}{|1 - h\lambda|}$. When $\lambda < 0$, the denominator $|1 - h\lambda| = 1 + h|\lambda| > 1$ for all $h > 0$, so the factor is always less than 1. This means Backward Euler is **unconditionally stable** (A-stable) — it never diverges, regardless of step size.

Backward Euler is still **first-order** accurate ($\mathcal{O}(h)$), same as Forward Euler.

> **Note.** For nonlinear equations, the implicit equation cannot be solved algebraically — it requires a **root-finding method** (e.g. Newton's method). We cover these techniques in the next chapter.

In [ ]:
# Complete code — just run this cell
# Backward Euler on the same fast-decay equation (algebraic formula for linear ODE)
lam = -10
t_be_fast = np.arange(t_span_fast[0], t_span_fast[1] + h_fast, h_fast)
y_be_fast = np.zeros(len(t_be_fast))
y_be_fast[0] = 1.0
for i in range(len(t_be_fast) - 1):
    y_be_fast[i + 1] = y_be_fast[i] / (1 - h_fast * lam)

plt.figure(figsize=(8, 5))
plt.plot(t_fast_dense, y_fast_exact(t_fast_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-10t}$")
plt.plot(t_fe_fast, y_fe_fast, "o-", color="crimson", markersize=5, alpha=0.7,
         label=f"Forward Euler (h={h_fast})")
plt.plot(t_be_fast, y_be_fast, "s-", color="forestgreen", markersize=5,
         label=f"Backward Euler (h={h_fast})")
plt.axhline(0, color="grey", linewidth=0.5)
plt.xlabel("t")
plt.ylabel("y")
plt.title("Backward Euler remains stable where Forward Euler fails")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2 Trapezoidal Method

Forward Euler uses the slope at the **start**. Backward Euler uses the slope at the **end**. The natural next idea: **average both slopes**.

$$y_{n+1} = y_n + \frac{h}{2}\Bigl[f(t_n,\, y_n) + f(t_{n+1},\, y_{n+1})\Bigr]$$

This is still implicit, but the payoff is significant: the Trapezoidal method is **second-order** ($\mathcal{O}(h^2)$) and **A-stable**.

**Algebraic solution for the test equation:**

$$y_{n+1} = y_n + \frac{h}{2}(\lambda y_n + \lambda y_{n+1}) \;\Longrightarrow\; y_{n+1} = y_n \cdot \frac{1 + h\lambda/2}{1 - h\lambda/2}$$

> **Note.** Like Backward Euler, the Trapezoidal method requires root-finding for nonlinear problems — covered in the next chapter.

In [ ]:
# Complete code — just run this cell
# Trapezoidal on the same fast-decay equation (algebraic formula for linear ODE)
t_tr_fast = np.arange(t_span_fast[0], t_span_fast[1] + h_fast, h_fast)
y_tr_fast = np.zeros(len(t_tr_fast))
y_tr_fast[0] = 1.0
for i in range(len(t_tr_fast) - 1):
    y_tr_fast[i + 1] = y_tr_fast[i] * (1 + h_fast * lam / 2) / (1 - h_fast * lam / 2)

plt.figure(figsize=(8, 5))
plt.plot(t_fast_dense, y_fast_exact(t_fast_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-10t}$")
plt.plot(t_fe_fast, y_fe_fast, "o-", color="crimson", markersize=5, alpha=0.7,
         label=f"Forward Euler (h={h_fast})")
plt.plot(t_tr_fast, y_tr_fast, "^-", color="darkorange", markersize=5,
         label=f"Trapezoidal (h={h_fast})")
plt.axhline(0, color="grey", linewidth=0.5)
plt.xlabel("t")
plt.ylabel("y")
plt.title("Trapezoidal method: stable and second-order accurate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 RK2 — Heun's Method

The Trapezoidal method achieves second-order accuracy but is implicit. Can we get the same accuracy with a purely **explicit** method? Yes — the idea is **predict, then correct**:

1. **Predict** using Forward Euler: $\tilde{y} = y_n + h \cdot f(t_n, y_n)$
2. **Correct** by averaging the slopes at both ends:

$$y_{n+1} = y_n + \frac{h}{2}\Bigl[f(t_n, y_n) + f(t_{n+1}, \tilde{y})\Bigr]$$

In the Runge–Kutta notation:

$$k_1 = f(t_n,\, y_n), \qquad k_2 = f(t_{n+1},\, y_n + h\,k_1), \qquad y_{n+1} = y_n + \frac{h}{2}(k_1 + k_2)$$

This is **Heun's method** — explicit (no iteration needed) and second-order.

In [ ]:
def rk2_heun(f, t_span, y0, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0

    # ===== YOUR CODE BELOW =====
    # Compute k1, k2, then update y.




    # ===== YOUR CODE ABOVE =====
    return t, y

In [ ]:
show_hint("rk2_heun")

In [ ]:
check_rk2_heun(rk2_heun)

### 2.4 RK2 — Midpoint Method

Another RK2 variant. Instead of averaging slopes at the two ends, the Midpoint method evaluates the slope at the **middle** of the interval:

$$k_1 = f(t_n,\, y_n), \qquad k_2 = f\!\left(t_n + \frac{h}{2},\; y_n + \frac{h}{2}\,k_1\right), \qquad y_{n+1} = y_n + h\,k_2$$

Like Heun's method, this is explicit and second-order. The two variants have similar accuracy; which one performs better depends on the specific problem.

In [ ]:
def rk2_midpoint(f, t_span, y0, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0

    # ===== YOUR CODE BELOW =====
    # Compute k1 at (t[i], y[i]), then k2 at the midpoint.




    # ===== YOUR CODE ABOVE =====
    return t, y

In [ ]:
show_hint("rk2_midpoint")

In [ ]:
check_rk2_midpoint(rk2_midpoint)

### 2.5 Grand Comparison

Let us put all five methods side by side on the running example ($h = 0.5$). Backward Euler and Trapezoidal are computed using their algebraic formulas for this linear ODE.

In [ ]:
# Complete code — just run this cell
h_comp = 0.5
lam_run = -2

# Student-implemented methods
t_fe, y_fe = forward_euler(f, t_span, y0, h_comp)
t_heun, y_heun = rk2_heun(f, t_span, y0, h_comp)
t_mid, y_mid = rk2_midpoint(f, t_span, y0, h_comp)

# Backward Euler (algebraic)
t_be = np.arange(t_span[0], t_span[1] + h_comp, h_comp)
y_be = np.zeros(len(t_be))
y_be[0] = y0
for i in range(len(t_be) - 1):
    y_be[i + 1] = y_be[i] / (1 - h_comp * lam_run)

# Trapezoidal (algebraic)
t_tr = np.arange(t_span[0], t_span[1] + h_comp, h_comp)
y_tr = np.zeros(len(t_tr))
y_tr[0] = y0
for i in range(len(t_tr) - 1):
    y_tr[i + 1] = y_tr[i] * (1 + h_comp * lam_run / 2) / (1 - h_comp * lam_run / 2)

plt.figure(figsize=(9, 5))
plt.plot(t_dense, y_exact(t_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-2t}$")
plt.plot(t_fe, y_fe, "o-", color="crimson", markersize=6, label="Forward Euler")
plt.plot(t_be, y_be, "s-", color="forestgreen", markersize=6, label="Backward Euler")
plt.plot(t_tr, y_tr, "^-", color="darkorange", markersize=6, label="Trapezoidal")
plt.plot(t_heun, y_heun, "D-", color="darkviolet", markersize=6, label="RK2 (Heun)")
plt.plot(t_mid, y_mid, "v-", color="saddlebrown", markersize=6, label="RK2 (Midpoint)")
plt.xlabel("t")
plt.ylabel("y")
plt.title(f"All five methods on $dy/dt = -2y$ (h = {h_comp})")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Method':<18s}  {'Type':<10s}  {'Order':<6s}  {'f calls/step'}")
print("-" * 55)
print(f"{'Forward Euler':<18s}  {'Explicit':<10s}  {'1':<6s}  {'1'}")
print(f"{'Backward Euler':<18s}  {'Implicit':<10s}  {'1':<6s}  {'—'}")
print(f"{'Trapezoidal':<18s}  {'Implicit':<10s}  {'2':<6s}  {'—'}")
print(f"{'RK2 (Heun)':<18s}  {'Explicit':<10s}  {'2':<6s}  {'2'}")
print(f"{'RK2 (Midpoint)':<18s}  {'Explicit':<10s}  {'2':<6s}  {'2'}")

### 2.6 Convergence

**Convergence** means that as the step size $h \to 0$, the numerical solution approaches the exact solution. The **order** tells us *how fast* it converges:

- A **first-order** method ($\mathcal{O}(h)$): halving $h$ halves the error.
- A **second-order** method ($\mathcal{O}(h^2)$): halving $h$ **quarters** the error.

On a **log–log plot** of error versus $h$, the slope equals the order. The plot below shows all five methods — first-order methods (Forward/Backward Euler) follow the slope-1 reference line, while second-order methods (Trapezoidal, RK2 Heun, RK2 Midpoint) follow slope 2.

In [ ]:
# Complete code — just run this cell
h_values = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]
lam_run = -2

def _be_algebraic(t_span, y0, h, lam):
    """Backward Euler using algebraic formula for dy/dt = lam*y."""
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i + 1] = y[i] / (1 - h * lam)
    return t, y

def _tr_algebraic(t_span, y0, h, lam):
    """Trapezoidal using algebraic formula for dy/dt = lam*y."""
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i + 1] = y[i] * (1 + h * lam / 2) / (1 - h * lam / 2)
    return t, y

methods = [
    ("Forward Euler",  "crimson",      "o"),
    ("Backward Euler", "forestgreen",  "s"),
    ("Trapezoidal",    "darkorange",   "^"),
    ("RK2 (Heun)",     "darkviolet",   "D"),
    ("RK2 (Midpoint)", "saddlebrown",  "v"),
]

plt.figure(figsize=(8, 5))
for name, col, mkr in methods:
    errs = []
    for hv in h_values:
        if name == "Forward Euler":
            t_m, y_m = forward_euler(f, t_span, y0, hv)
        elif name == "Backward Euler":
            t_m, y_m = _be_algebraic(t_span, y0, hv, lam_run)
        elif name == "Trapezoidal":
            t_m, y_m = _tr_algebraic(t_span, y0, hv, lam_run)
        elif name == "RK2 (Heun)":
            t_m, y_m = rk2_heun(f, t_span, y0, hv)
        else:
            t_m, y_m = rk2_midpoint(f, t_span, y0, hv)
        idx = np.argmin(np.abs(t_m - 1.0))
        errs.append(abs(y_m[idx] - y_exact(t_m[idx])))
    plt.loglog(h_values, errs, marker=mkr, color=col, label=name)

# Reference slope lines
h_ref = np.array(h_values)
plt.loglog(h_ref, 0.3 * h_ref, "--", color="grey", alpha=0.5, label="Slope 1")
plt.loglog(h_ref, 0.5 * h_ref**2, ":", color="grey", alpha=0.5, label="Slope 2")
plt.xlabel("Step size h")
plt.ylabel("Error at t = 1")
plt.title("Convergence: first-order vs second-order methods")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Summary

| Method | Type | Order | Stability |
|--------|------|-------|-----------|
| Forward Euler | Explicit | 1 | Conditional ($h < 2/|\lambda|$) |
| Backward Euler | Implicit | 1 | Unconditional (A-stable) |
| Trapezoidal | Implicit | 2 | Unconditional (A-stable) |
| RK2 (Heun) | Explicit | 2 | Conditional |
| RK2 (Midpoint) | Explicit | 2 | Conditional |

**Key takeaways:**

- **Explicit methods** are simple to implement but have restricted stability regions.
- **Implicit methods** are unconditionally stable but require solving an equation at each step — for nonlinear problems this means root-finding (next chapter).
- **Accuracy (order) and stability are independent** — a higher-order explicit method does not inherit better stability.
- Real-world solvers like `scipy.integrate.solve_ivp` automatically choose between explicit and implicit methods based on the problem.